# Análisis de Estabilidad Temporal y Regional
## Tesis: Benchmarking Explainable Gradient Boosting and Tabular Deep Learning
## for Predicting Satisfaction with Democracy in Latin America (1995–2024)

**Objetivo:** Responder PI3 y OE3 — ¿los determinantes identificados son
estables entre splits históricos y subregiones geográficas?

Requiere haber ejecutado los notebooks 02 y 04 previamente.

### Estructura
| Sección | Contenido |
|---|---|
| 1–2 | Importaciones y configuración |
| 3 | Correlación Spearman entre rankings SHAP (test/test/test) |
| 4 | Heatmap y bump chart de estabilidad |
| 5 | Análisis de estabilidad por subregión |
| 6 | MAE ordinal por país y split |
| 7 | Prueba formal de H4 y H5 |
| 8 | Resumen y guardado |

## 1. Importaciones

In [1]:
import sys
sys.path.append("..")

import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json

warnings.filterwarnings("ignore")

from utils.config import (
    setup_plots, THEME, PATHS, SPLIT, SUBREGIONES, PAISES_EXCLUIR_EVAL,
    PAISES_EXCLUIR_EVAL, COL_TARGET, COL_PAIS,
    ETIQUETAS, ETIQUETAS_FEATURES, BLOQUES, bloque_de,
)
from utils.io import (
    cargar_pipeline, cargar_split_parquet, cargar_resultados,
    cargar_shap_values, shap_disponible, cargar_mejor_modelo,
)
from utils.plots import (
    plot_heatmap_estabilidad, plot_bump_chart,
    plot_spearman_estabilidad, plot_shap_por_subregion,
    plot_rendimiento_por_pais, save_figure, model_color,
)
from utils.metrics import evaluar
from sklearn.metrics import mean_absolute_error

setup_plots()
print("✓ Importaciones completadas.")

✓ Importaciones completadas.


## 2. Configuración

In [2]:
# =============================================================================
# Parámetros — NB05: Análisis por subregiones y estrategias de balanceo
# Propósito rediseñado:
#   - Análisis de rendimiento por subregión (responde a solicitud del tutor)
#   - Análisis de SHAP por subregión (¿qué explica la satisfacción por región?)
#   - Comparativa SMOTE-NC vs. pesos de clase en clases minoritarias
# =============================================================================
from utils.io import cargar_pipeline, cargar_resultados, cargar_shap_values, cargar_split_parquet
from sklearn.metrics import mean_absolute_error, cohen_kappa_score, accuracy_score

MODELO_PRINCIPAL = "CatBoost"    # actualizar tras ver resultados de NB03
ESTRATEGIA_PPAL  = "pesos_clase" # mejor estrategia de E1

setup_plots()
print(f"Modelo principal  : {MODELO_PRINCIPAL}")
print(f"Estrategia ppal   : {ESTRATEGIA_PPAL}")
print(f"Test set          : {SPLIT['test']}")
print(f"Países en test    : 16 (sin Venezuela ni Nicaragua)")
print(f"Subregiones       : {list(SUBREGIONES.keys())}")


Modelo principal  : CatBoost
Estrategia ppal   : pesos_clase
Test set          : [2023, 2024]
Países en test    : 16 (sin Venezuela ni Nicaragua)
Subregiones       : ['Cono Sur', 'Región Andina', 'Brasil', 'Centroamérica', 'México y Caribe']


## 3. Correlación Spearman entre rankings SHAP

La correlación de Spearman entre los rankings de importancia SHAP de
distintos splits mide directamente la estabilidad de los determinantes.

- **r = 1.0**: los mismos factores explican la satisfacción en todos los períodos
- **r < 0.7**: cambio sustancial en los determinantes entre períodos
- **r < 0.5**: cambio estructural (umbral para H5)

In [3]:
# =============================================================================
# Cargar resultados y datos del split de prueba
# =============================================================================
df_res = cargar_resultados(split="test")
df_te  = cargar_split_parquet("test")

# Cargar artefacto del modelo principal
art   = cargar_pipeline(MODELO_PRINCIPAL, ESTRATEGIA_PPAL)
feats = art["features"]

print(f"Test set cargado: {len(df_te):,} registros")
print(f"Países: {sorted(df_te[COL_PAIS].unique()) if COL_PAIS in df_te.columns else 'N/A'}")


Test set cargado: 655 registros
Países: N/A


## 4. Visualizaciones de estabilidad temporal

In [4]:
# =============================================================================
# Análisis de rendimiento por subregión — modelo principal
# =============================================================================
if COL_PAIS not in df_te.columns:
    print("⚠ Columna pais_nombre no disponible en el test parquet")
else:
    from utils.preprocessing import construir_split
    import numpy as np

    # Predicciones sobre el test completo
    if art["tipo_modelo"] in ("olo","tabnet"):
        vars_cat = [c for c in art.get("vars_categoricas",[]) if c in df_te.columns]
        cols_num = [c for c in feats if c not in vars_cat]
        X_num = pd.DataFrame(art["imp_num"].transform(df_te[feats][cols_num]),
                             columns=cols_num, index=df_te.index)
        if vars_cat and art.get("imp_cat"):
            X_cat = pd.DataFrame(art["imp_cat"].transform(df_te[feats][vars_cat]),
                                 columns=vars_cat, index=df_te.index)
            X_imp = pd.concat([X_num, X_cat], axis=1)[feats]
        else:
            X_imp = X_num
        cols_sc = [c for c in feats if c not in vars_cat]
        X_sc = X_imp.copy()
        X_sc[cols_sc] = art["scaler"].transform(X_imp[cols_sc])
        y_pred_all = art["modelo"].predict(X_sc.values)
    else:
        y_pred_all = art["modelo"].predict(df_te[feats])
        if hasattr(y_pred_all, "flatten"):
            y_pred_all = y_pred_all.flatten()
    y_pred_all = np.array(y_pred_all).astype(int)
    y_true_all = df_te[COL_TARGET].values.astype(int)

    filas_sr = []
    for sr_name, paises_sr in SUBREGIONES.items():
        mask = df_te[COL_PAIS].isin(paises_sr)
        if mask.sum() == 0:
            continue
        y_t = y_true_all[mask.values]
        y_p = y_pred_all[mask.values]
        mae  = mean_absolute_error(y_t, y_p)
        kappa = cohen_kappa_score(y_t, y_p, weights="quadratic") if len(set(y_t))>1 else float("nan")
        acc  = accuracy_score(y_t, y_p)
        mejor_pais = None; peor_pais = None
        min_mae = 999; max_mae = -1
        for p in paises_sr:
            mp = df_te[COL_PAIS] == p
            if mp.sum() < 5:
                continue
            mae_p = mean_absolute_error(y_true_all[mp.values], y_pred_all[mp.values])
            if mae_p < min_mae: min_mae = mae_p; mejor_pais = p
            if mae_p > max_mae: max_mae = mae_p; peor_pais  = p
        filas_sr.append({"subregion":sr_name, "n_registros":mask.sum(),
                          "mae_ordinal":round(mae,4), "kappa_cuadratico":round(kappa,4),
                          "accuracy":round(acc,4), "mejor_pais":mejor_pais,
                          "peor_pais":peor_pais})
        print(f"  {sr_name:<22}: MAE={mae:.4f}  κ={kappa:.4f}  Acc={acc:.4f}")

    df_sr = pd.DataFrame(filas_sr)
    df_sr.to_csv(PATHS["FOLDER_RESULTS_TABLES"] / "mae_subregiones.csv", index=False)
    print()
    print("✓ Guardado: results/tables/mae_subregiones.csv")
    print("  → Tabla 5.9 de la tesis: MAE por subregión")


⚠ Columna pais_nombre no disponible en el test parquet


In [5]:
# =============================================================================
# SHAP medio por subregión — top 5 variables por región
# =============================================================================
if not shap_disponible(MODELO_PRINCIPAL):
    print("\u26a0 Ejecutar NB04 primero para calcular SHAP")
elif COL_PAIS not in df_te.columns:
    print("\u26a0 pais_nombre no disponible en test.parquet \u2014 agregar fix en NB02")
else:
    df_shap_all = cargar_shap_values(MODELO_PRINCIPAL)
    df_te_r = df_te.reset_index(drop=True)

    dict_sr = {}
    for sr_name, paises_sr in SUBREGIONES.items():
        mask = df_te_r[COL_PAIS].isin(paises_sr)
        if mask.sum() < 20:
            continue
        dict_sr[sr_name] = df_shap_all[mask].abs().mean()

    if dict_sr:
        df_shap_sr = pd.DataFrame(dict_sr)
        print("Top 5 variables SHAP por subregión:")
        for sr_name in df_shap_sr.columns:
            top5 = df_shap_sr[sr_name].sort_values(ascending=False).head(5).index.tolist()
            etiq = [ETIQUETAS_FEATURES.get(v, v) for v in top5]
            print(f"  {sr_name:<22}: {etiq}")
        plot_shap_por_subregion(df_shap_sr, nombre_archivo="05_shap_por_subregion")
    else:
        print("\u26a0 Sin subregiones con datos suficientes (mín. 20 registros)")


⚠ Ejecutar NB04 primero para calcular SHAP


## 5. Análisis de estabilidad por subregión

In [6]:
# =============================================================================
# Comparativa E1: Rendimiento en clases minoritarias por estrategia
# ¿SMOTE-NC mejora el F1 de la clase 0 (Para nada satisfecho)?
# =============================================================================
df_test_res = cargar_resultados(split="test")
df_ord = df_test_res[df_test_res["variante_target"]=="ordinal_4clases"].copy()

if not df_ord.empty:
    from sklearn.metrics import classification_report
    print("F1 de clase 0 (Para nada satisfecho) por estrategia:")
    print("(Calculado sobre las predicciones del conjunto de prueba)")
    print()
    for est in ["sin_balanceo","pesos_clase","smotenc"]:
        sub = df_ord[df_ord["estrategia_balanceo"]==est]
        if sub.empty:
            print(f"  {est}: sin resultados")
            continue
        mejor = sub.loc[sub["kappa_cuadratico"].idxmax()]
        print(f"  {est:<15}: Kappa={mejor['kappa_cuadratico']:.4f}  "
              f"F1_macro={mejor['f1_macro']:.4f}  Acc={mejor['accuracy']:.4f}")

    df_ord.to_csv(PATHS["FOLDER_RESULTS_TABLES"] / "comparativa_estrategias_balanceo.csv", index=False)
    print()
    print("✓ Guardado: results/tables/comparativa_estrategias_balanceo.csv")
    print("  → Tabla 5.X de la tesis: Comparativa E1")
else:
    print("⚠ Sin resultados de E1 — ejecutar NB02")


F1 de clase 0 (Para nada satisfecho) por estrategia:
(Calculado sobre las predicciones del conjunto de prueba)

  sin_balanceo   : Kappa=0.5385  F1_macro=0.4879  Acc=0.5450
  pesos_clase    : Kappa=0.5250  F1_macro=0.4896  Acc=0.5176
  smotenc        : Kappa=0.5348  F1_macro=0.4977  Acc=0.5542

✓ Guardado: results/tables/comparativa_estrategias_balanceo.csv
  → Tabla 5.X de la tesis: Comparativa E1


## 6. MAE ordinal por país y split

In [7]:
# =============================================================================
# Análisis de rendimiento por país para las 3 estrategias de balanceo
# Métrica: MAE ordinal (robusta ante clases minoritarias)
# =============================================================================
import joblib

df_te_mae = cargar_split_parquet("test")
filas_mae = []

for estrat in ["sin_balanceo", "pesos_clase", "smotenc"]:
    if COL_PAIS not in df_te_mae.columns:
        print("\u26a0 pais_nombre no disponible \u2014 agregar fix en NB02")
        break

    y_te = df_te_mae[COL_TARGET].astype(int).values

    for modelo in ["OLO", "XGBoost", "CatBoost", "LightGBM", "TabNet"]:
        try:
            art_m   = cargar_pipeline(modelo, estrat)
            tipo    = art_m["tipo_modelo"]
            feats_m = art_m["features"]
            X_te_m  = df_te_mae[[f for f in feats_m if f in df_te_mae.columns]]
            X_te_m  = X_te_m.reindex(columns=feats_m)

            if tipo in ("olo", "tabnet"):
                vars_cat_m = [c for c in art_m.get("vars_categoricas", []) if c in X_te_m.columns]
                cols_num_m = [c for c in feats_m if c not in vars_cat_m]
                X_num_m = pd.DataFrame(art_m["imp_num"].transform(X_te_m[cols_num_m]),
                                       columns=cols_num_m, index=X_te_m.index)
                if vars_cat_m and art_m.get("imp_cat") is not None:
                    X_cat_m = pd.DataFrame(art_m["imp_cat"].transform(X_te_m[vars_cat_m]),
                                           columns=vars_cat_m, index=X_te_m.index)
                    X_imp_m = pd.concat([X_num_m, X_cat_m], axis=1)[feats_m]
                else:
                    X_imp_m = X_num_m
                cols_sc_m = [c for c in feats_m if c not in vars_cat_m]
                X_sc_m = X_imp_m.copy()
                X_sc_m[cols_sc_m] = art_m["scaler"].transform(X_imp_m[cols_sc_m])
                y_raw = art_m["modelo"].predict(
                    X_sc_m.values if tipo == "olo" else X_sc_m.values.astype(np.float32))
            else:
                X_in = X_te_m.copy()
                if modelo == "CatBoost":
                    for col in art_m.get("vars_categoricas", []):
                        if col in X_in.columns:
                            X_in[col] = X_in[col].fillna(-999).astype(int).astype(str)
                y_raw = art_m["modelo"].predict(X_in)

            y_pred = y_raw.flatten() if hasattr(y_raw, "flatten") else y_raw

            for pais in df_te_mae[COL_PAIS].unique():
                if pais in PAISES_EXCLUIR_EVAL:
                    continue
                mask = df_te_mae[COL_PAIS] == pais
                if mask.sum() < 10:
                    continue
                mae = mean_absolute_error(y_te[mask.values], y_pred[mask.values])
                sr  = next((s for s, ps in SUBREGIONES.items() if pais in ps), "Sin clasificar")
                filas_mae.append({
                    "modelo": modelo, "estrategia_balanceo": estrat,
                    "pais": pais, "subregion": sr,
                    "mae_ordinal": mae, "n": int(mask.sum()),
                })
        except FileNotFoundError:
            pass
        except Exception as e:
            print(f"  \u26a0 {modelo} {estrat}: {e}")

df_mae_completo = pd.DataFrame(filas_mae)
if not df_mae_completo.empty:
    print("MAE Ordinal promedio por modelo y estrategia:")
    pivot_mae = (df_mae_completo.groupby(["modelo", "estrategia_balanceo"])["mae_ordinal"]
                 .mean().unstack().round(4))
    print(pivot_mae.to_string())
    df_mae_completo.to_csv(
        PATHS["FOLDER_RESULTS_TABLES"] / "mae_por_pais_todos.csv", index=False)
    print("\n\u2713 Tabla MAE completa guardada")
else:
    print("\u26a0 Sin datos MAE \u2014 verificar pipelines y pais_nombre en test.parquet")


⚠ pais_nombre no disponible — agregar fix en NB02
⚠ Sin datos MAE — verificar pipelines y pais_nombre en test.parquet


In [8]:
# Heatmap: MAE por subregión × estrategia para el modelo principal
if not df_mae_completo.empty:
    df_sr_sp = (df_mae_completo[df_mae_completo["modelo"] == MODELO_PRINCIPAL]
                .groupby(["subregion", "estrategia_balanceo"])["mae_ordinal"].mean()
                .unstack())

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(df_sr_sp, annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.3, ax=ax,
                cbar_kws={"label": "MAE Ordinal"})
    ax.set_title(f"MAE Ordinal por subregión y estrategia \u2014 {MODELO_PRINCIPAL}",
                 fontweight="bold")
    save_figure(f"05_mae_subregion_estrategia_{MODELO_PRINCIPAL}")
    plt.show()


## 7. Prueba formal de H4 y H5

**H4:** la importancia de las variables varía entre subregiones geográficas,
con corrupción y confianza institucional mostrando mayor variación regional.

**H5:** la correlación Spearman entre los rankings de importancia SHAP de
distintas subregiones es alta (r \u2265 0.7), indicando que los determinantes
identificados son robustos y generalizables en toda América Latina.

> **Nota metodológica:** con el diseño de split único (test: 2023\u20132024),
> la estabilidad temporal no puede analizarse entre períodos. Se evalúa
> en su lugar la estabilidad *regional*, comparando los rankings SHAP
> calculados para cada subregión del conjunto de prueba.


In [9]:
# =============================================================================
# Prueba de H4 y H5: estabilidad de determinantes entre subregiones
# Con diseño de split único la estabilidad temporal no aplica; se evalúa
# la estabilidad regional comparando rankings SHAP entre subregiones.
# =============================================================================
if not shap_disponible(MODELO_PRINCIPAL):
    print("\u26a0 Ejecutar NB04 primero para calcular SHAP")
elif COL_PAIS not in df_te.columns:
    print("\u26a0 pais_nombre no disponible \u2014 agregar fix en NB02")
else:
    df_shap_all = cargar_shap_values(MODELO_PRINCIPAL)
    df_te_r = df_te.reset_index(drop=True)

    dict_sr = {}
    for sr_name, paises_sr in SUBREGIONES.items():
        mask = df_te_r[COL_PAIS].isin(paises_sr)
        if mask.sum() < 20:
            continue
        dict_sr[sr_name] = df_shap_all[mask].abs().mean()

    if len(dict_sr) < 2:
        print("\u26a0 Insuficientes subregiones con datos SHAP (mín. 2 con \u226520 registros)")
    else:
        df_shap_sr = pd.DataFrame(dict_sr)

        print("=" * 60)
        print("PRUEBA DE H4")
        print("=" * 60)
        print("Variación de importancia SHAP por bloque entre subregiones:")
        print()
        for bloque, vars_b in BLOQUES.items():
            vars_pres = [v for v in vars_b if v in df_shap_sr.index]
            if not vars_pres:
                continue
            imp_bloque = df_shap_sr.loc[vars_pres].mean()
            rango   = imp_bloque.max() - imp_bloque.min()
            sr_max  = imp_bloque.idxmax()
            sr_min  = imp_bloque.idxmin()
            print(f"  {bloque:<42}: rango={rango:.4f}  "
                  f"mayor en '{sr_max}'  menor en '{sr_min}'")

        print()
        print("=" * 60)
        print("PRUEBA DE H5")
        print("=" * 60)
        print("Correlación Spearman entre rankings SHAP de subregiones:")
        print()

        srs = df_shap_sr.columns.tolist()
        n_sr = len(srs)
        corr_matrix = pd.DataFrame(index=srs, columns=srs, dtype=float)
        for sri in srs:
            for srj in srs:
                r, _ = stats.spearmanr(df_shap_sr[sri], df_shap_sr[srj])
                corr_matrix.loc[sri, srj] = round(r, 4)

        print(corr_matrix.to_string())
        print()

        vals_tri = [corr_matrix.loc[srs[i], srs[j]]
                    for i in range(n_sr) for j in range(i)]
        r_min  = min(vals_tri) if vals_tri else float("nan")
        r_mean = sum(vals_tri) / len(vals_tri) if vals_tri else float("nan")

        print(f"  Correlación mínima : {r_min:.4f}")
        print(f"  Correlación media  : {r_mean:.4f}")
        print()

        if r_min >= 0.7:
            print("  \u2713 H5 CONFIRMADA: r_min \u2265 0.7 \u2014 los determinantes son")
            print("    robustos entre subregiones de América Latina.")
        elif r_min >= 0.5:
            print("  ~ H5 PARCIALMENTE CONFIRMADA: 0.5 \u2264 r_min < 0.7 \u2014")
            print("    los determinantes muestran estabilidad moderada.")
        else:
            print("  \u2717 H5 NO CONFIRMADA: r_min < 0.5 \u2014 existe cambio")
            print("    estructural entre subregiones.")

        plot_spearman_estabilidad(
            corr_matrix.astype(float),
            titulo=f"Correlación Spearman entre rankings SHAP \u2014 {MODELO_PRINCIPAL}",
            nombre_archivo="05_spearman_subregiones"
        )

        corr_matrix.to_csv(PATHS["FOLDER_RESULTS_TABLES"] / "spearman_subregiones.csv")
        print("\u2713 Guardado: results/tables/spearman_subregiones.csv")


⚠ Ejecutar NB04 primero para calcular SHAP


## 8. Resumen y guardado

In [10]:
print("=" * 60)
print("RESUMEN \u2014 Análisis de estabilidad temporal y regional")
print("=" * 60)
print(f"  Modelo analizado : {MODELO_PRINCIPAL}")
print(f"  Estrategia       : {ESTRATEGIA_PPAL}")
print(f"  Split de prueba  : test (2023\u20132024)")
print()
print("Archivos generados:")
for f in sorted(PATHS["FOLDER_RESULTS_FIGURES"].glob("05_*.png")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*spearman*")):
    print(f"  {f.name}")
for f in sorted(PATHS["FOLDER_RESULTS_TABLES"].glob("*mae*")):
    print(f"  {f.name}")


RESUMEN — Análisis de estabilidad temporal y regional
  Modelo analizado : CatBoost
  Estrategia       : pesos_clase
  Split de prueba  : test (2023–2024)

Archivos generados:
  tabla_maestra_xai_CatBoost.csv
